# AMLAnnotator Tutorial

This tutorial demonstrates how to use AMLAnnotator to classify cells in your AML scRNA-seq data into:
1. **Malignant vs Normal** cells
2. **Blast group** (among malignant cells)
3. **LSC type** (among High_LSC_score_shared cells)

## Installation

```bash
pip install git+https://github.com/sinakane/AMLAnnotator.git
```

## 1. Load your data

AMLAnnotator works with any AnnData object containing scRNA-seq data. Your data should have:
- Standard gene symbols in `var_names`
- Raw counts (either in `.X` or a named layer like `'raw_counts'`)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Load your AML scRNA-seq data
adata = sc.read_h5ad("your_data.h5ad")
print(f"Loaded {adata.n_obs} cells x {adata.n_vars} genes")

## 2. Initialize AMLAnnotator

The annotator loads pre-trained models automatically. No additional data files are needed.

In [ ]:
from amlannotator import AMLAnnotator

annotator = AMLAnnotator()

# View model information
info = annotator.get_model_info()
for level, details in info.items():
    print(f"\n{level}")
    print(f"  Classes: {details['classes']}")
    print(f"  Features: {details['n_features']}")
    print(f"  Accuracy: {details['accuracy']:.4f}")

## 3. Annotate

Run the full hierarchical annotation pipeline with a single call.

**Parameters:**
- `layer` — Layer containing raw counts (e.g. `'raw_counts'`). If `None`, uses `.X`.
- `copy` — If `True`, return a copy instead of modifying in place.
- `malignant_prob_threshold` — Cells classified as normal but with an averaged malignant probability above this threshold are reclassified as malignant. Default `0.3`. This recovers borderline malignant cells that transcriptionally resemble healthy progenitors. Set to `0.5` to disable reclassification.

In [ ]:
# Annotate — specify the layer containing raw counts
adata = annotator.annotate(adata, layer="raw_counts")

# If raw counts are in .X:
# adata = annotator.annotate(adata)

# Adjust the threshold to recover more borderline malignant cells:
# adata = annotator.annotate(adata, layer="raw_counts", malignant_prob_threshold=0.2)

# Disable reclassification entirely:
# adata = annotator.annotate(adata, layer="raw_counts", malignant_prob_threshold=0.5)

## 4. Explore results

### 4.1 Summary

In [ ]:
# Level 1: Malignant vs Normal
print("=== Level 1: Malignant vs Normal ===")
print(adata.obs["aml_malignant_normal"].value_counts())
print(f"\nMean confidence: {adata.obs['aml_malignant_confidence'].mean():.3f}")

# Level 2: Blast Group
print("\n=== Level 2: Blast Group ===")
print(adata.obs["aml_blast_group"].value_counts())

# Level 3: LSC Type
print("\n=== Level 3: LSC Type ===")
print(adata.obs["aml_lsc_type"].value_counts())

### 4.2 UMAP visualisation

In [ ]:
# If you already have a UMAP embedding
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc.pl.umap(adata, color="aml_malignant_normal", ax=axes[0], show=False,
           title="Level 1: Malignant vs Normal")
sc.pl.umap(adata, color="aml_blast_group", ax=axes[1], show=False,
           title="Level 2: Blast Group")
sc.pl.umap(adata, color="aml_lsc_type", ax=axes[2], show=False,
           title="Level 3: LSC Type")

plt.tight_layout()
plt.show()

### 4.3 Confidence scores

Each cell receives a confidence score (0-1) representing the maximum averaged probability across the ensemble models. Higher scores indicate more agreement between classifiers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc.pl.umap(adata, color="aml_malignant_confidence", ax=axes[0], show=False,
           title="L1 Confidence", cmap="viridis")
sc.pl.umap(adata, color="aml_blast_confidence", ax=axes[1], show=False,
           title="L2 Confidence", cmap="viridis")
sc.pl.umap(adata, color="aml_lsc_confidence", ax=axes[2], show=False,
           title="L3 Confidence", cmap="viridis")

plt.tight_layout()
plt.show()

### 4.4 Per-class probabilities

You can also access the per-class probability for each cell. These are the averaged predicted probabilities across all three ensemble models.

In [ ]:
# Example: probability of being malignant
sc.pl.umap(adata, color="aml_prob_malignant", cmap="Reds",
           title="P(malignant)")

### 4.5 Filter by confidence

You can filter cells by confidence threshold to retain only high-confidence annotations.

In [ ]:
# Keep only cells with high confidence at Level 1
high_conf = adata[adata.obs["aml_malignant_confidence"] > 0.8]
print(f"High-confidence cells: {high_conf.n_obs} / {adata.n_obs}")
print(high_conf.obs["aml_malignant_normal"].value_counts())

## 5. Save results

In [ ]:
# Save annotated object
adata.write("my_data_annotated.h5ad")

# Or export annotations as CSV
cols = [c for c in adata.obs.columns if c.startswith("aml_")]
adata.obs[cols].to_csv("aml_annotations.csv")
print(f"Saved {len(cols)} annotation columns")